**Build a Simple LLM Application with LCEL**
In this quickstart we'll show you hoq to build a simple LLM application with Langchain. This application will translate text from English to another language. This is relatively simple LLM application - it's just a single LLM call plus some prompting. Still. this is a great way to get started with LangChain - a lot of features can be built with just some prompting and an LLM call.
* Using language models
* Using PromptTemplate and OutputParser
* Using LangChain Expression Language (LCEL) to chain components together
* Debugging and tracing your application using LangSmith
* Deploying you application with LangServe

In [3]:
### Open AI API key and Open source models-- llama3, Gemma3,mistral--Groq

import os
from dotenv import load_dotenv
load_dotenv()

import openai
#openai.api_key=os.getenv("GROQ_API_KEY")

groq_api_key=os.getenv("GROQ_API_KEY")


In [4]:
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI

model=ChatGroq(model="llama-3.3-70b-versatile", api_key=groq_api_key)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x10f7d4ad0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10f7d57f0>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [10]:
from langchain_core.messages import HumanMessage,SystemMessage

messages=[
    SystemMessage(content="Translate the following from English to French"),
    HumanMessage(content="Hello, how are you?")
]


In [11]:
response=model.invoke(messages)

In [12]:
from  langchain_core.output_parsers import StrOutputParser
parser=StrOutputParser()
parser.invoke(response)

'Bonjour, comment allez-vous ?'

In [13]:
### using LCEL - chain the components
chain=model|parser
chain.invoke(messages)

'Bonjour, comment allez-vous ?'

In [14]:
### Prompt Template
from langchain_core.prompts import ChatPromptTemplate
generic_template="Translate the following into {language}."
prompt=ChatPromptTemplate.from_messages(
    [
        ("system",generic_template),
        ("user","{text}")
    ]
)


In [20]:
result=prompt.invoke({"language":"Hindi","text":"Hello, how are you?"})

In [21]:
result.to_messages()

[SystemMessage(content='Translate the following into Hindi.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={})]

In [28]:
### Chaining together components with LCEL
chain=prompt|model|parser
chain.invoke({"language":"Urdu","text":"Hello, how are you?"})

'اسلام علیکم، آپ کیسے ہیں؟'

ChatPromptTemplate(input_variables=['language', 'text'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['language'], input_types={}, partial_variables={}, template='Translate the following into {language}.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='{text}'), additional_kwargs={})])
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': Fals